# Sovereign Workbench — Agent-Protocol LoRA

**SIH26117 · Sovereign On-Premise Agentic AI Workbench (MRPL)**

Fine-tunes **`Qwen/Qwen3.5-4B`** (Apache 2.0) so it reliably speaks our agent
protocol: one JSON object per turn, the right tool, correctly formed arguments.

## Why this and not something else

The workbench already supplies the knowledge and the arithmetic — RAG retrieves
the SOP clause, the sandbox runs the calculation. What is left for the model is
narrow and mechanical: **choose a tool and format the call**. That is the part a
small model gets wrong, it is the single biggest live-demo risk, and it is
exactly what a LoRA fixes cheaply.

We are *not* training in refinery knowledge. Facts come from the corpus at
inference time, with citations. A model that memorised thresholds would be worse
— it could state one without a source.

## The sovereignty rule this notebook obeys

Training on a rented GPU is fine; the proposal permits HF GPUs for development.
What must never happen is the **runtime** calling out. So this notebook ends by
producing **GGUF weights + an Ollama Modelfile** that run entirely on the demo
machine. Nothing here becomes a runtime dependency.

## What you need

- A GPU with ≥24 GB VRAM (A10G / L4 / A100). 4-bit QLoRA also fits in 16 GB.
- `data/training/agent_sft.jsonl`, produced by `scripts/build_training_data.py`.

## Order of cells

1. Environment check → 2. Install → 3. Get the dataset → 4. Inspect it →
5. Load the model → 6. LoRA config → 7. **Baseline eval** → 8. Train →
9. **Post-train eval** → 10. Merge → 11. GGUF + Modelfile → 12. Ship it back

## 1 · Environment check

Stop early if there is no GPU, rather than 40 minutes in.

In [ ]:
import subprocess, sys, platform

print("python :", sys.version.split()[0])
print("system :", platform.platform())

try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as exc:
    print(f"nvidia-smi unavailable: {exc}")
    print("\nThis notebook needs a GPU. On HF, pick a GPU Space or a GPU-backed notebook.")

## 2 · Install

Pinned deliberately. `trl` and `peft` move fast and their `SFTConfig` argument
names change between minor versions; an unpinned install is the most common way
this notebook breaks months later.

In [ ]:
%pip install -q --upgrade pip
%pip install -q "torch>=2.4" "transformers>=4.57" "trl>=0.23" "peft>=0.17" "datasets>=3.0" "accelerate>=1.0" "bitsandbytes>=0.44" sentencepiece protobuf

import torch, transformers, trl, peft
print("torch       ", torch.__version__, "| cuda:", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("trl         ", trl.__version__)
print("peft        ", peft.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"gpu          {props.name}  {props.total_memory/1e9:.0f} GB")

## 3 · Get the dataset

Three ways, in order of preference:

1. **Clone the repo** and regenerate — guarantees the training prompt matches
   the prompt `src/core/prompts.py` sends at inference. If those two drift, the
   fine-tune teaches a format the runtime never uses and buys you nothing.
2. **Upload** `data/training/agent_sft.jsonl` next to this notebook.
3. Pull it from a HF dataset repo you pushed earlier.

In [ ]:
import os, json
from pathlib import Path

REPO_URL  = os.environ.get("SOVEREIGN_REPO", "https://github.com/hs-zz27/sih.git")
DATA_PATH = Path("agent_sft.jsonl")

if not DATA_PATH.exists():
    if Path("sih").exists() or os.system(f"git clone --depth 1 {REPO_URL} sih") == 0:
        # Regenerate from the live prompt + tool registry.
        rc = os.system("cd sih && pip install -q pyyaml pydantic numpy && "
                       "python scripts/build_training_data.py --n 800")
        candidate = Path("sih/data/training/agent_sft.jsonl")
        if rc == 0 and candidate.exists():
            DATA_PATH.write_text(candidate.read_text(encoding="utf-8"), encoding="utf-8")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "No agent_sft.jsonl. Upload it, or set SOVEREIGN_REPO to a reachable clone URL."
    )

rows = [json.loads(line) for line in DATA_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"{len(rows)} examples loaded from {DATA_PATH}")

## 4 · Inspect before you train

Read one example end to end. Most bad fine-tunes are visible here.

In [ ]:
example = rows[0]
for message in example["messages"]:
    print("=" * 78)
    print(message["role"].upper())
    print("=" * 78)
    print(message["content"][:1200])
    print()

# Sanity: every assistant turn must be a single valid JSON object, because that
# is precisely what we are teaching.
bad = 0
for row in rows:
    for message in row["messages"]:
        if message["role"] == "assistant":
            try:
                json.loads(message["content"])
            except json.JSONDecodeError:
                bad += 1
print(f"malformed assistant turns: {bad}  (must be 0)")
assert bad == 0

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(rows).train_test_split(test_size=0.05, seed=42)
print(dataset)

## 5 · Load `Qwen/Qwen3.5-4B` in 4-bit

QLoRA: frozen 4-bit base, small trainable adapters. Fits comfortably and trains
in minutes on this dataset size.

**Thinking mode.** Qwen3.5 emits `<think>…</think>` before its answer by
default. We train on non-thinking traces because the demo runs with thinking
disabled for latency. The agent's parser strips the tags regardless, so turning
thinking back on later cannot break parsing.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "Qwen/Qwen3.5-4B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # correct for training; flip to left for batched generation

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",
)
model.config.use_cache = False          # incompatible with gradient checkpointing
print(model.config.model_type, f"{model.num_parameters()/1e9:.2f}B params")

In [ ]:
# Confirm the chat template renders our conversation as expected. If this looks
# wrong, everything downstream is wrong.
rendered = tokenizer.apply_chat_template(
    dataset["train"][0]["messages"], tokenize=False, add_generation_prompt=False
)
print(rendered[:1500])

lengths = [
    len(tokenizer.apply_chat_template(r["messages"], tokenize=True))
    for r in dataset["train"].select(range(min(200, len(dataset["train"]))))
]
print(f"\ntokens per example: mean {sum(lengths)//len(lengths)}, max {max(lengths)}")
MAX_SEQ_LEN = 4096
print("truncated at 4096:", sum(1 for n in lengths if n > MAX_SEQ_LEN))

## 6 · LoRA configuration

`r=16` is ample for a formatting-and-selection task; we are shaping output
structure, not installing new knowledge. Targeting attention **and** MLP
projections gives the adapter enough capacity to change tool-choice behaviour,
not just phrasing.

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
print(peft_config)

## 7 · Baseline eval — measure before you change anything

This is the cell that makes the exercise honest. We ask the **untrained** model
to produce agent turns and score them with the workbench's own parser:

- **parse rate** — did we get one valid decision object?
- **valid tool rate** — is the named tool one that actually exists?

If the baseline is already ~100%, the LoRA is unnecessary and you should say so
rather than train anyway. Record these two numbers; they are the honest before/
after for the pitch.

In [ ]:
import re, json

# The workbench's own parsing rules, inlined so this cell works without the repo.
_THINK = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)
_OPEN  = re.compile(r"<think>.*$",        re.DOTALL | re.IGNORECASE)
_FENCE = re.compile(r"```(?:json)?\s*(.*?)```", re.DOTALL)

TOOLS = {"read_file", "write_file", "list_files", "search_documents", "run_python"}


def parse_decision(text: str):
    """Mirror of src/core/agent.py::_parse_decision. Returns dict or None."""
    if not text or not text.strip():
        return None
    text = _OPEN.sub("", _THINK.sub("", text)).strip()

    candidates = []
    if text.startswith("{"):
        candidates.append(text)
    candidates += [m.group(1).strip() for m in _FENCE.finditer(text)]

    depth, start, in_str, esc = 0, -1, False, False
    for i, ch in enumerate(text):
        if in_str:
            if esc: esc = False
            elif ch == "\\": esc = True
            elif ch == '"': in_str = False
            continue
        if ch == '"': in_str = True
        elif ch == "{":
            if depth == 0: start = i
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and start >= 0:
                candidates.append(text[start:i + 1]); start = -1
            elif depth < 0: depth = 0

    for candidate in candidates:
        try:
            payload = json.loads(candidate)
        except json.JSONDecodeError:
            continue
        if isinstance(payload, dict) and ("tool" in payload or "final_answer" in payload):
            return payload
    return None


@torch.no_grad()
def generate_turns(model, tokenizer, rows, limit=40, max_new_tokens=192):
    """Generate the model's next turn for each held-out example."""
    model.eval()
    tokenizer.padding_side = "left"
    outputs = []
    for row in rows[:limit]:
        messages = row["messages"]
        # Cut at the first assistant turn: ask the model to produce it.
        cut = next(i for i, m in enumerate(messages) if m["role"] == "assistant")
        prompt = tokenizer.apply_chat_template(
            messages[:cut], tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        generated = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False,                      # greedy: matches demo settings
            pad_token_id=tokenizer.pad_token_id,
        )
        text = tokenizer.decode(generated[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        outputs.append((text, json.loads(messages[cut]["content"])))
    tokenizer.padding_side = "right"
    return outputs


def score(outputs):
    parsed_ok = valid_tool = tool_match = 0
    for text, expected in outputs:
        decision = parse_decision(text)
        if decision is None:
            continue
        parsed_ok += 1
        tool = decision.get("tool")
        if tool is None or tool in TOOLS:
            valid_tool += 1
        if tool == expected.get("tool"):
            tool_match += 1
    n = len(outputs)
    return {
        "n": n,
        "parse_rate": round(parsed_ok / n, 3),
        "valid_tool_rate": round(valid_tool / n, 3),
        "tool_match_rate": round(tool_match / n, 3),
    }


held_out = list(dataset["test"])
baseline_outputs = generate_turns(model, tokenizer, held_out)
BASELINE = score(baseline_outputs)
print("BASELINE:", BASELINE)
print("\n--- sample generation ---\n", baseline_outputs[0][0][:600])

## 8 · Train

`assistant_only_loss=True` means loss is computed on the model's turns alone.
Without it the model also learns to predict our tool observations — wasted
capacity on text it will never have to produce.

In [ ]:
from trl import SFTConfig, SFTTrainer

OUTPUT_DIR = "qwen35-4b-sovereign-agent-lora"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,          # effective batch 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    bf16=True,
    max_length=MAX_SEQ_LEN,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    report_to="none",                       # no telemetry, on principle
    seed=42,
)

# assistant_only_loss landed in trl 0.20; degrade gracefully on older versions.
try:
    sft_config.assistant_only_loss = True
except Exception as exc:
    print("assistant_only_loss unavailable, training on full sequences:", exc)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    processing_class=tokenizer,
)

trainer.model.print_trainable_parameters()

In [ ]:
train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("\ntrain loss:", round(train_result.training_loss, 4))
print("adapter saved to:", OUTPUT_DIR)

## 9 · Post-train eval

Same held-out set, same scorer. This is the number that justifies the work.

In [ ]:
trainer.model.config.use_cache = True
trained_outputs = generate_turns(trainer.model, tokenizer, held_out)
TRAINED = score(trained_outputs)

print(f"{'metric':<20} {'baseline':>10} {'trained':>10} {'delta':>10}")
print("-" * 52)
for key in ("parse_rate", "valid_tool_rate", "tool_match_rate"):
    before, after = BASELINE[key], TRAINED[key]
    print(f"{key:<20} {before:>10.3f} {after:>10.3f} {after - before:>+10.3f}")

print("\n--- sample generation ---\n", trained_outputs[0][0][:600])

if TRAINED["parse_rate"] <= BASELINE["parse_rate"]:
    print("\nNo improvement in parse rate. Say so rather than shipping the adapter:")
    print("the base model may already be reliable enough, in which case skip the LoRA.")

## 10 · Merge the adapter

The demo runs one set of weights in Ollama, so the adapter is merged into the
base model rather than loaded separately.

In [ ]:
import gc, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

MERGED_DIR = "qwen35-4b-sovereign-agent-merged"

del trainer, model
gc.collect(); torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, dtype=torch.bfloat16, device_map="cpu", trust_remote_code=True,
)
merged = PeftModel.from_pretrained(base, OUTPUT_DIR).merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

print("merged weights ->", MERGED_DIR)

## 11 · GGUF + Ollama Modelfile — the part that keeps the claim true

Converts to GGUF and writes the Modelfile the demo machine uses. After this the
workbench needs **nothing** from HuggingFace at runtime.

`Q4_K_M` is the right quantisation for a 4B model on a laptop: ~2.5 GB, minimal
quality loss, fast enough that the step trace animates rather than crawls.

In [ ]:
import os
from pathlib import Path

if not Path("llama.cpp").exists():
    os.system("git clone --depth 1 https://github.com/ggerganov/llama.cpp")
os.system("pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt")

GGUF_F16 = "sovereign-agent-f16.gguf"
GGUF_Q4  = "sovereign-agent-q4_k_m.gguf"

os.system(f"python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {GGUF_F16} --outtype f16")

# Quantise. Newer llama.cpp builds the tool via cmake; fall back if absent.
if os.system(f"./llama.cpp/build/bin/llama-quantize {GGUF_F16} {GGUF_Q4} Q4_K_M") != 0:
    os.system("cd llama.cpp && cmake -B build -DLLAMA_CURL=OFF && cmake --build build --config Release -j --target llama-quantize")
    os.system(f"./llama.cpp/build/bin/llama-quantize {GGUF_F16} {GGUF_Q4} Q4_K_M")

for path in (GGUF_F16, GGUF_Q4):
    if Path(path).exists():
        print(f"{path}: {Path(path).stat().st_size/1e9:.2f} GB")

In [ ]:
MODELFILE = f"""FROM ./{GGUF_Q4}

# Sovereign Workbench agent model - Qwen3.5-4B + agent-protocol LoRA (merged).
# Runs entirely on the demo machine. No external service is contacted.

PARAMETER temperature 0
PARAMETER top_p 0.8
PARAMETER top_k 20
PARAMETER num_ctx 8192
PARAMETER repeat_penalty 1.05

# The agent supplies its own system prompt per task type (document / code /
# general), so none is baked in here.
"""

Path("Modelfile").write_text(MODELFILE, encoding="utf-8")
print(MODELFILE)

## 12 · Ship it back to the demo machine

Download `sovereign-agent-q4_k_m.gguf` and `Modelfile`, put them in one folder,
then **on the demo laptop**:

```bash
ollama create sovereign-agent -f Modelfile
ollama run sovereign-agent "hello"
```

Then point the workbench at it — `config.yaml`, and nothing else:

```yaml
models:
  document: "sovereign-agent"
  code:     "sovereign-agent"
  general:  "sovereign-agent"
```

Verify end to end with the network off:

```bash
python -m pytest tests -q
curl -s http://127.0.0.1:8000/api/health | python -m json.tool
```

### Optional: push the adapter to the Hub

For versioning during development only. The demo never fetches it.

In [ ]:
PUSH = False   # set True to publish the adapter for versioning

if PUSH:
    from huggingface_hub import login, HfApi
    login()  # or set HF_TOKEN
    repo_id = "your-username/qwen35-4b-sovereign-agent-lora"
    HfApi().create_repo(repo_id, exist_ok=True)
    HfApi().upload_folder(folder_path=OUTPUT_DIR, repo_id=repo_id)
    print("pushed:", repo_id)
else:
    print("PUSH is False - nothing uploaded. The demo does not need the Hub.")

## Record the result

Write the before/after numbers into `HARDCODED.md` (or the pitch notes) with the
date, the dataset size and the commit that produced the data. If the LoRA did
not improve the parse rate, say that plainly and ship the base model — an honest
null result is defensible in Q&A, and an unexplained adapter is not.